<!--nav--> [🗺 Learning path](README.md) · **35/48** · ◀ [GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb) · [Modern GPU & Model Architecture](./Modern_GPU_And_Model_Architecture.ipynb) ▶

# Measuring GPU Code Honestly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Measuring_GPU_Code_Honestly.ipynb)

[GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb) is about making
code fast. This one is about knowing whether you did.

That sounds like the easier half. It is not. Almost every wrong GPU benchmark I have seen is
wrong for one of six mechanical reasons, none of which involve the kernel at all — and every
one of them biases the number in the flattering direction:

| # | the mistake | what it does to your number |
|---|---|---|
| 1 | host clock around an async launch | measures the *enqueue*, ~5 µs regardless of the kernel |
| 2 | no warmup | first call pays context setup, JIT, autotuning, allocator |
| 3 | hot cache between reps | a memory-bound kernel reports L2 bandwidth, 3–5x too high |
| 4 | mean over a few reps | one preemption moves the headline |
| 5 | launch overhead inside the window | for a 5 µs kernel, most of the number |
| 6 | no denominator | 400 GB/s is excellent on a T4 and a catastrophe on an H100 |

The fixes are all mechanical too, and they live in
[`tools/gpu_bench.py`](https://github.com/sugeerth/gpu-training-notebooks/blob/main/tools/gpu_bench.py)
in this repo. This notebook is the argument for each one, with the effect measured rather than
asserted.

**Runs anywhere.** Mistakes 4 and 6 — the statistics and the denominator — are demonstrated on
whatever machine you are on, because they are not GPU-specific and are the two that transfer
to everything else you will ever measure. Mistake 3 has a CPU analogue that shows the same
effect at the same magnitude. With a GPU you get all six.

In [ ]:
# Setup. On Colab this clones the repo.
import os, subprocess, sys, statistics, math, random
from pathlib import Path

def find_repo():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "tools" / "gpu_bench.py").exists():
            return cand
    return None

URL = "https://github.com/sugeerth/gpu-training-notebooks"
BRANCH = "claude/serving-optimization-notebooks-jyerty"   # until this lands on main

REPO = find_repo()
if REPO is None:
    dest = Path("/content/gpu-training-notebooks")
    if not (dest / "tools" / "gpu_bench.py").exists():
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", URL, str(dest)], check=True)
        if not (dest / "tools" / "gpu_bench.py").exists():
            subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd=str(dest))
            subprocess.run(["git", "checkout", "FETCH_HEAD"], cwd=str(dest))
    REPO = dest
sys.path.insert(0, str(REPO / "tools"))
import gpu_bench
from gpu_bench import Bench, Result

b = Bench()
print(b.describe())
HAVE_GPU = b.dev.has_cuda

## Part 1 · The clock

A CUDA kernel launch is **asynchronous**. `kernel<<<...>>>()` puts work on a stream and
returns; the GPU may not have started yet. So this, the most natural thing to write, measures
the wrong thing entirely:

```python
t0 = time.perf_counter()
model(x)                      # returns immediately
t1 = time.perf_counter()      # the GPU is still working
```

You get the enqueue cost — a few microseconds, roughly constant, *independent of how much work
you queued*. The two ways out:

- `torch.cuda.synchronize()` before both readings. Correct, but it also waits for anything
  else on the device, and the sync itself costs a few microseconds.
- **CUDA events**, which are timestamps recorded *in the stream*, by the GPU, in GPU clock
  ticks. They measure exactly the interval between two points in the queue and nothing else.
  This is what `Bench.time()` uses.

In [ ]:
if HAVE_GPU:
    import torch
    n = 1 << 26                                     # 256 MB
    x = torch.randn(n, device="cuda")
    y = torch.empty_like(x)
    nbytes = 2 * x.numel() * x.element_size()

    wrong = b.wall_time(lambda: y.copy_(x), "wall clock, no sync")
    right = b.time(lambda: y.copy_(x), "cuda events", nbytes=nbytes)
    print(b.table([wrong, right]))
    print(f"\nThe host clock is reporting {right.median_ms / wrong.median_ms:.0f}x too fast,")
    print("and it would report the same number for a kernel ten times the size.")
    print(f"Implied bandwidth from the wrong number: {nbytes/(wrong.median_ms/1e3)/1e9:,.0f} GB/s")
    print(f"— which is more than the card has, and that is the tell.")
else:
    print("""No GPU here, so there is no asynchrony to demonstrate. On an A100 this cell prints
roughly:

kernel                        median ms     ±MAD      p95     GB/s   timer
------------------------------------------------------------------------------
wall clock, no sync              0.0071   0.0004   0.0092      -     wall clock, no sync
cuda events                      0.2790   0.0012   0.2831    481.0   cuda-events

The host clock is reporting 39x too fast.
Implied bandwidth from the wrong number: 18,900 GB/s — nine times what the card has.

That last line is the check worth internalising: if your measured bandwidth exceeds the
card's spec, you did not discover anything. You measured the queue.""")

### The sanity check that costs nothing

> If your number implies more bandwidth or more FLOP/s than the hardware has, the number is
> wrong. Always. There is no clever kernel that beats the memory controller.

This one check catches mistakes 1, 3 and 5 at once, and it requires only that you know your
card's two numbers. That is the entire reason every table in
[`kernels/`](https://github.com/sugeerth/gpu-training-notebooks/tree/main/kernels) prints a
percentage next to the rate.

## Part 2 · Warmup

The first call into any GPU code path pays for things no subsequent call pays for: CUDA
context creation (~100 ms), module loading and JIT of the kernel for your specific
architecture, cuBLAS/cuDNN autotuning picking an algorithm for your shapes, and a caching
allocator that has not yet cached anything.

Against a kernel that runs for 300 µs, a 30 ms first call is a **100x** error. And it is not a
constant offset you can subtract — it depends on what ran before.

In [ ]:
if HAVE_GPU:
    import torch, time
    torch.cuda.empty_cache()
    m = 4096
    A = torch.randn(m, m, device="cuda")
    B = torch.randn(m, m, device="cuda")

    per_call = []
    for i in range(8):
        s, e = torch.cuda.Event(True), torch.cuda.Event(True)
        s.record(); torch.mm(A, B); e.record(); e.synchronize()
        per_call.append(s.elapsed_time(e))
    print("first eight calls to the same matmul, in milliseconds:")
    for i, t in enumerate(per_call):
        print(f"  call {i}: {t:8.3f} ms" + ("   <- what a no-warmup benchmark reports" if i == 0 else ""))
    steady = statistics.median(per_call[3:])
    print(f"\nsteady state {steady:.3f} ms; the first call is {per_call[0]/steady:.1f}x that.")
    print("Bench.time() discards 10 calls before it starts recording.")
    del A, B; torch.cuda.empty_cache()
else:
    print("""On an A100 with a cold context, the first eight calls to a 4096-cube matmul look
like:

  call 0:   41.812 ms   <- what a no-warmup benchmark reports
  call 1:    2.190 ms
  call 2:    2.181 ms
  ...
  steady state 2.18 ms; the first call is 19.2x that.

The size of the effect depends on what else has already initialised, which is exactly why it
cannot be corrected for after the fact — only discarded.""")

## Part 3 · The cache you forgot about

Run the same kernel 100 times on the same input and, after the first iteration, the input is
in L2 — which on an H100 is 50 MB with roughly 2x the bandwidth of HBM, and on an A100 40 MB.
A memory-bound kernel benchmarked on a 32 MB array is not measuring memory at all.

This is the most common way a genuine speedup turns out not to exist: the microbenchmark fits
in cache, the production workload does not.

The fix is to write an L2-sized buffer between repetitions, which `Bench(flush_l2=True)` does
by default. The demonstration below works on **any** machine, because a CPU's L2/L3 does
exactly the same thing for exactly the same reason.

In [ ]:
# The cache effect, on whatever machine you are on. numpy, no GPU required.
import numpy as np, time

def copy_bandwidth(mb, target_bytes=3e8):
    """Median bandwidth of a repeated copy. Each size gets the same total work, so the
    small sizes are not dominated by per-call overhead — the mistake that makes a cache
    sweep look non-monotonic and get dismissed."""
    n = max(1024, int(mb * 1e6 // 8))
    src, dst = np.ones(n, dtype=np.float64), np.empty(n, dtype=np.float64)
    inner = max(1, int(target_bytes // (2 * n * 8)))
    for _ in range(2):
        for _ in range(inner):
            np.copyto(dst, src)
    ts = []
    for _ in range(7):
        t0 = time.perf_counter()
        for _ in range(inner):
            np.copyto(dst, src)
        ts.append((time.perf_counter() - t0) / inner)
    return 2 * n * 8 / statistics.median(ts) / 1e9

sizes = [(0.25, "L2"), (2, "L2/L3"), (16, "L3"), (128, "L3/DRAM"), (512, "DRAM")]
print(f"{'working set':>14} {'GB/s':>9}   probably in")
measured = []
for mb, where in sizes:
    gbps = copy_bandwidth(mb)
    measured.append(gbps)
    print(f"{mb:>11.2f} MB {gbps:>9.1f}   {where}")

print(f"""
Same instruction, same code, same everything: {max(measured)/min(measured):.1f}x apart on this
machine, depending only on whether the data was already close by. On a server CPU with a large
L3 the spread is wider; on a GPU, L2 is 40-50 MB and roughly 2x HBM bandwidth, so a benchmark
whose working set fits in it reports a number the card cannot sustain.

The rule: size your benchmark's working set to the *production* working set, or flush the
cache between reps. Doing neither is how a speedup that does not exist gets published.""")

## Part 4 · The statistic

GPU timing distributions are not Gaussian. They are a tight mode plus a right tail from clock
throttling, other tenants on the device, driver interrupts and host scheduling. Summarising
that with mean ± stddev is the wrong tool twice over: the mean chases the tail, and the stddev
is dominated by it.

Use **median** for the headline and **MAD** (median absolute deviation) for the spread, then
report the tail *separately* as p95 if the tail is what you care about — for serving latency
it usually is, and for kernel throughput it usually is not.

The cell below builds a distribution with a realistic shape and shows what each summary says
about it.

In [ ]:
# One realistic latency distribution, five summaries. Runs anywhere.
random.seed(7)
samples = [random.gauss(1.00, 0.01) for _ in range(94)]        # the mode: 1% jitter
samples += [random.gauss(1.6, 0.15) for _ in range(5)]         # clock throttle events
samples += [4.3]                                               # one preemption
random.shuffle(samples)

mean = statistics.mean(samples)
med = statistics.median(samples)
mad = statistics.median([abs(s - med) for s in samples])
sd = statistics.pstdev(samples)
p95 = sorted(samples)[int(0.95 * (len(samples) - 1))]

print(f"n = {len(samples)}  (94 clean, 5 throttled, 1 preempted)\n")
print(f"  mean   ± stddev   {mean:6.3f} ± {sd:5.3f} ms")
print(f"  median ± MAD      {med:6.3f} ± {mad:5.3f} ms")
print(f"  p95               {p95:6.3f} ms")
print(f"  min               {min(samples):6.3f} ms")
print(f"""
The kernel takes 1.00 ms. The median says 1.00. The mean says {mean:.2f} — {100*(mean/med-1):.0f}% high,
entirely because of 6 samples out of 100, and it would say something different every run.

Reporting the *minimum* is the opposite error and just as common: it is the one repetition
where nothing went wrong, which is not a number you will ever see again in production.

A histogram beats all of them. If you only ever adopt one habit from this notebook, plot the
distribution once before you trust any summary of it.""")

# ASCII histogram — no matplotlib dependency, works in any environment.
lo, hi, nb_ = min(samples), max(samples), 28
width = (hi - lo) / nb_
counts = [0] * nb_
for s in samples:
    counts[min(nb_ - 1, int((s - lo) / width))] += 1
print("\ndistribution:")
for i, c in enumerate(counts):
    if c:
        print(f"  {lo + i*width:5.2f} ms | {'#' * c}{'' if c else ''} {c}")

In [ ]:
# The same reasoning applied to a real measurement, if you have a GPU.
if HAVE_GPU:
    import torch
    n = 1 << 24
    x = torch.randn(n, device="cuda"); y = torch.empty_like(x)
    r = b.time(lambda: y.copy_(x), "copy 64MB", nbytes=2 * n * 4, reps=200)
    lo95, hi95 = r.ci95_ms
    print(f"median      {r.median_ms:.4f} ms")
    print(f"MAD         {r.mad_ms:.4f} ms")
    print(f"95% CI      [{lo95:.4f}, {hi95:.4f}] ms   <- how many digits you may quote")
    print(f"p95         {r.p95_ms:.4f} ms")
    print(f"min         {r.min_ms:.4f} ms")
    print(f"\nbandwidth   {r.gbps:.0f} GB/s = {100*r.gbps/b.dev.measured_gbps:.0f}% of "
          f"this card's measured ceiling")
    digits = max(0, -int(math.floor(math.log10(max(1e-9, hi95 - lo95)))))
    print(f"\nThe confidence interval is {hi95-lo95:.4f} ms wide, so quoting more than "
          f"{digits} decimal places is fiction.")
else:
    print("""(GPU section.) The point it makes: Bench.time() reports a bootstrap 95% interval
for the median, and that interval tells you how many digits you are entitled to quote. A
'1.2847 ms' with a 0.03 ms interval is four digits of theatre around one digit of signal —
and it is how two configurations come to look different when they are not.""")

## Part 5 · Launch overhead, and CUDA graphs

Every kernel launch costs the *host* a few microseconds of driver work. That is invisible for
a 10 ms kernel and dominant for a 5 µs one — and a decode step is made of hundreds of small
kernels, so this is not a corner case, it is the LLM inference case.

Two consequences:

1. **Your microbenchmark of a small kernel is mostly measuring the launch.** So is the
   production workload, which is the honest defence of measuring it that way — as long as you
   know that is what you are looking at.
2. **CUDA graphs remove it.** Capture a sequence of launches once, replay the whole thing with
   a single call. vLLM, TensorRT-LLM and torch.compile's `mode="reduce-overhead"` all do this
   for exactly this reason, and it is worth 10–30% of a decode step on small models.

`Bench.time(..., use_graph=True)` captures and replays, so you can measure both the launch-
bound and the launch-free version of the same work and see the gap.

The catch, and it is the same catch the serving engines hit: nothing inside the captured region
may synchronize, allocate new memory, or depend on a CPU value. That is why engines have a
fixed set of graph-captured batch sizes and fall back to eager execution for anything else.

In [ ]:
if HAVE_GPU:
    import torch
    # 60 tiny kernels in sequence — the shape of a decode step, not of a benchmark.
    xs = [torch.randn(4096, device="cuda") for _ in range(4)]
    out = torch.empty_like(xs[0])

    def chain():
        t = xs[0]
        for i in range(60):
            t = torch.nn.functional.relu(t * 1.0001 + xs[i % 4])
        out.copy_(t)

    eager = b.time(chain, "60 small kernels, eager", reps=50)
    try:
        graphed = b.time(chain, "60 small kernels, graph", reps=50, use_graph=True)
        print(b.table([eager, graphed], baseline=0))
        saved = eager.median_ms - graphed.median_ms
        print(f"\nlaunch overhead removed: {saved*1e3:.0f} us over 60 kernels "
              f"= {saved*1e3/60:.1f} us per launch")
        print("That per-launch figure is the one to carry around. Multiply it by the number of")
        print("kernels in your decode step to see what graph capture is worth to you.")
    except Exception as exc:
        print(b.table([eager]))
        print(f"\ngraph capture unavailable here: {exc}")
else:
    print("""(GPU section.) On an A100 this typically shows ~5 us per launch, so a decode step
built from 300 small kernels carries ~1.5 ms of pure driver overhead — against a step time
that might be 8 ms. Graph capture takes most of it back, which is why
Anatomy_Of_A_Decode_Step.ipynb treats "overhead" as its own slice of the budget.""")

## Part 6 · The denominator

A time is not a measurement. A time divided by a ceiling is.

Most roofline plots use the **vendor peak**, which assumes clocks no sustained workload holds
and, for bandwidth, an efficiency no memory controller reaches. Drawn that way, a perfect
kernel lands at 80% and everyone learns that 80% means "done". Drawn against what a
*saturating* kernel actually achieves on this card in this process, a good kernel lands near
100% and a bad one has nowhere to hide.

That is why `Bench` measures its own ceilings at construction: a large device-to-device copy
for bandwidth, and a large matmul for FLOP/s, in both fp32 and fp16.

In [ ]:
if HAVE_GPU:
    import torch
    print(b.describe())
    print()
    d = b.dev
    print(f"An fp32 kernel on this card is memory-bound below {d.ridge_fp32:.0f} FLOP/byte;")
    print(f"an fp16 tensor-core kernel is memory-bound below {d.ridge_fp16:.0f}.")
    print("\nWhere the LLM inference kernels sit:\n")
    ops = [("decode GEMV, fp16 weights",   2 / 2),
           ("decode GEMV, int4 weights",   2 / 0.5),
           ("decode attention, fp16 KV",   0.5),
           ("RMSNorm",                     0.25),
           ("SwiGLU (elementwise)",        0.25),
           ("prefill GEMM, 512 tokens",    2 * 512 / (2 + 2 * 512 / 4096)),
           ("training GEMM, 8192 tokens",  400.0)]
    for name, ai in ops:
        bound = "memory" if ai < d.ridge_fp16 else "compute"
        print(f"  {name:<28} {ai:8.2f} FLOP/byte  -> {bound}-bound")
    print("""
Everything above the last two lines is memory-bound, which is the single fact that organises
all of LLM serving: at decode time more FLOPs are free and fewer bytes is the only lever.""")
else:
    print("""(GPU section.) Without a card there is no ceiling to measure, so here is the
structure of the argument instead:

  ridge point = peak FLOP/s / peak bytes/s

  A100:  19.5 TF fp32 / 2039 GB/s =  10 FLOP/byte     (312 TF fp16 -> 153)
  H100:  67   TF fp32 / 3350 GB/s =  20 FLOP/byte     (990 TF fp16 -> 296)
  L4:    30   TF fp32 /  300 GB/s = 101 FLOP/byte     (121 TF fp16 -> 403)

and where the inference kernels sit:

  decode GEMV, fp16 weights     1.0 FLOP/byte   memory-bound
  decode GEMV, int4 weights     4.0             memory-bound
  decode attention              0.5             memory-bound
  RMSNorm / SwiGLU              0.25            memory-bound
  prefill GEMM, 512 tokens    ~400              compute-bound

Every decode-time kernel is one to three orders of magnitude below the ridge. That is why the
whole serving track is about bytes.""")

## Part 7 · The checklist

Before you believe a GPU benchmark — yours or anyone's:

1. **Was it timed with CUDA events or after a synchronize?** If neither, it is the enqueue cost.
2. **Was there a warmup?** How many iterations, and was the context already alive?
3. **Does the working set exceed L2?** If not, was the cache flushed between reps?
4. **Median or mean?** Over how many reps? Is the distribution shown anywhere?
5. **How many kernels per timed region?** If it is one small kernel, most of the number is the
   launch — and a CUDA-graph number is the fairer comparison.
6. **What fraction of the hardware's ceiling is it?** If the answer implies more bandwidth than
   the card has, stop.
7. **Is it correct?** A fast wrong kernel is the easiest thing in this field to ship, which is
   why every program in `kernels/` checks its output against a CPU reference on the same run
   it is timing, and exits non-zero if it disagrees.

Point 7 is the one that gets skipped. Point 3 is the one that produces the most exciting wrong
results.

### Apply it to something

The programs in [`kernels/`](https://github.com/sugeerth/gpu-training-notebooks/tree/main/kernels)
are built to satisfy all seven. Run them and read the output against the list:

In [ ]:
# Run the kernel suite and read its output against the checklist above.
import subprocess
p = subprocess.run("make --no-print-directory 01_copy", shell=True, cwd=str(REPO / "kernels"),
                   capture_output=True, text=True)
print(p.stdout or p.stderr)
print("""
Checklist, line by line, for that table:
  1. CUDA events                      -> common.cuh, bench::time_kernel
  2. 10 warmup iterations discarded   -> same function
  3. L2-sized buffer zeroed each rep  -> bench::L2Flusher
  4. median + MAD over 50 reps        -> bench::summarize
  5. one kernel per region, stated    -> and the launch cost is visible in the small cases
  6. % of achievable bandwidth        -> the last column, vs 90% of the card's peak
  7. checked against a CPU reference  -> the 'max err' column, and the exit code""")

## What to read next

- [GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb) — the kernels
  this notebook measures, and the hardware they are written for.
- [The Hardware Roofline](./Hardware_Roofline_NVIDIA_vs_AMD.ipynb) — the ridge point in full,
  and how the same argument runs on AMD.
- [Benchmark & Capacity Planning](./Serving_Benchmark_Capacity_Planning.ipynb) — the same
  discipline one level up: measuring a *server* rather than a kernel, where the equivalent
  mistakes are closed-loop load generators and latency measured from admission rather than
  arrival.
- [Anatomy of a Decode Step](./Anatomy_Of_A_Decode_Step.ipynb) — where these microseconds
  actually land in one token.

The `tools/gpu_bench.py` module is importable on its own, and running it directly
(`python tools/gpu_bench.py`) prints a report for whatever hardware it finds.